This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch torchvision datasets --upgrade -q

## Fundamentals of machine learning

### Generalization: The goal of machine learning

#### Underfitting and overfitting

##### Noisy training data

##### Ambiguous features

##### Rare features and spurious correlations

In [ ]:
# PyTorch: load MNIST via torchvision instead of keras.datasets; expose the raw
# uint8 NumPy arrays so the shapes/dtypes match what the chapter expects.
from torchvision.datasets import MNIST
import numpy as np

mnist_train = MNIST(root="./data", train=True, download=True)
train_images, train_labels = mnist_train.data.numpy(), mnist_train.targets.numpy()
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255

train_images_with_noise_channels = np.concatenate(
    [train_images, np.random.random((len(train_images), 784))], axis=1
)

train_images_with_zeros_channels = np.concatenate(
    [train_images, np.zeros((len(train_images), 784))], axis=1
)

In [ ]:
import torch
from torch import nn

# PyTorch: no compile()/fit(). We define a small reusable training loop that
# mirrors Keras: it carves out a validation split, runs mini-batch SGD, and
# returns a history dict shaped like Keras\' history.history
# ({"loss", "accuracy", "val_loss", "val_accuracy"}) so the plotting cells below
# keep working unchanged.
def fit_classifier(
    model,
    x,
    y,
    epochs,
    batch_size,
    optimizer,
    loss_fn,
    validation_split=0.0,
    binary=False,
):
    x = torch.as_tensor(np.asarray(x), dtype=torch.float32)
    if binary:
        y = torch.as_tensor(np.asarray(y), dtype=torch.float32).view(-1, 1)
    else:
        y = torch.as_tensor(np.asarray(y), dtype=torch.long)

    n = len(x)
    n_val = int(n * validation_split)
    # Keras takes the LAST fraction of the data as validation (no shuffling).
    x_val, y_val = x[n - n_val :], y[n - n_val :]
    x_train, y_train = x[: n - n_val], y[: n - n_val]

    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

    def metrics(model, xb, yb):
        with torch.no_grad():
            out = model(xb)
            loss = loss_fn(out, yb).item()
            if binary:
                pred = (torch.sigmoid(out) > 0.5).float()
                acc = (pred == yb).float().mean().item()
            else:
                acc = (out.argmax(dim=1) == yb).float().mean().item()
        return loss, acc

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(x_train))
        for i in range(0, len(x_train), batch_size):
            idx = perm[i : i + batch_size]
            inputs, targets = x_train[idx], y_train[idx]
            out = model(inputs)
            loss = loss_fn(out, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        tr_loss, tr_acc = metrics(model, x_train, y_train)
        history["loss"].append(tr_loss)
        history["accuracy"].append(tr_acc)
        if n_val > 0:
            v_loss, v_acc = metrics(model, x_val, y_val)
            history["val_loss"].append(v_loss)
            history["val_accuracy"].append(v_acc)
            print(
                f"Epoch {epoch + 1}/{epochs}: loss {tr_loss:.4f} acc {tr_acc:.4f} "
                f"val_loss {v_loss:.4f} val_acc {v_acc:.4f}"
            )
        else:
            print(f"Epoch {epoch + 1}/{epochs}: loss {tr_loss:.4f} acc {tr_acc:.4f}")

    # PyTorch: return an object exposing .history like a Keras History.
    return type("History", (), {"history": history})()


# PyTorch: nn.Linear needs explicit in_features; the inputs here have 1568
# columns (784 image pixels + 784 extra channels). We drop the final softmax
# because nn.CrossEntropyLoss expects raw logits.
def get_model(in_features):
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, 10),
    )


model = get_model(train_images_with_noise_channels.shape[1])
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
history_noise = fit_classifier(
    model,
    train_images_with_noise_channels,
    train_labels,
    epochs=10,
    batch_size=128,
    optimizer=optimizer,
    loss_fn=loss_fn,
    validation_split=0.2,
)

model = get_model(train_images_with_zeros_channels.shape[1])
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
history_zeros = fit_classifier(
    model,
    train_images_with_zeros_channels,
    train_labels,
    epochs=10,
    batch_size=128,
    optimizer=optimizer,
    loss_fn=loss_fn,
    validation_split=0.2,
)

In [0]:
import matplotlib.pyplot as plt

val_acc_noise = history_noise.history["val_accuracy"]
val_acc_zeros = history_zeros.history["val_accuracy"]
epochs = range(1, 11)
plt.plot(
    epochs,
    val_acc_noise,
    "b-",
    label="Validation accuracy with noise channels",
)
plt.plot(
    epochs,
    val_acc_zeros,
    "r--",
    label="Validation accuracy with zeros channels",
)
plt.title("Effect of noise channels on validation accuracy")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Accuracy")
plt.legend()
plt.show()

#### The nature of generalization in deep learning

In [ ]:
mnist_train = MNIST(root="./data", train=True, download=True)
train_images, train_labels = mnist_train.data.numpy(), mnist_train.targets.numpy()
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255

random_train_labels = train_labels[:]
np.random.shuffle(random_train_labels)

# PyTorch: drop the final softmax; CrossEntropyLoss works on raw logits.
model = nn.Sequential(
    nn.Linear(28 * 28, 512),
    nn.ReLU(),
    nn.Linear(512, 10),
)
optimizer = torch.optim.RMSprop(model.parameters())
loss_fn = nn.CrossEntropyLoss()
fit_classifier(
    model,
    train_images,
    random_train_labels,
    epochs=100,
    batch_size=128,
    optimizer=optimizer,
    loss_fn=loss_fn,
    validation_split=0.2,
)

##### The manifold hypothesis

##### Interpolation as a source of generalization

##### Why deep learning works

##### Training data is paramount

### Evaluating machine-learning models

#### Training, validation, and test sets

##### Simple hold-out validation

##### K-fold validation

##### Iterated K-fold validation with shuffling

#### Beating a common-sense baseline

#### Things to keep in mind about model evaluation

### Improving model fit

#### Tuning key gradient descent parameters

In [ ]:
mnist_train = MNIST(root="./data", train=True, download=True)
train_images, train_labels = mnist_train.data.numpy(), mnist_train.targets.numpy()
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255

model = nn.Sequential(
    nn.Linear(28 * 28, 512),
    nn.ReLU(),
    nn.Linear(512, 10),
)
optimizer = torch.optim.RMSprop(model.parameters(), lr=1.0)
loss_fn = nn.CrossEntropyLoss()
fit_classifier(
    model, train_images, train_labels, epochs=10, batch_size=128,
    optimizer=optimizer, loss_fn=loss_fn, validation_split=0.2,
)

In [ ]:
model = nn.Sequential(
    nn.Linear(28 * 28, 512),
    nn.ReLU(),
    nn.Linear(512, 10),
)
optimizer = torch.optim.RMSprop(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()
fit_classifier(
    model, train_images, train_labels, epochs=10, batch_size=128,
    optimizer=optimizer, loss_fn=loss_fn, validation_split=0.2,
)

#### Using better architecture priors

#### Increasing model capacity

In [ ]:
# PyTorch: a single linear layer mapping pixels -> 10 logits (softmax dropped).
model = nn.Sequential(nn.Linear(28 * 28, 10))
optimizer = torch.optim.RMSprop(model.parameters())
loss_fn = nn.CrossEntropyLoss()
history_small_model = fit_classifier(
    model, train_images, train_labels, epochs=20, batch_size=128,
    optimizer=optimizer, loss_fn=loss_fn, validation_split=0.2,
)

In [0]:
import matplotlib.pyplot as plt

val_loss = history_small_model.history["val_loss"]
epochs = range(1, 21)
plt.plot(epochs, val_loss, "b-", label="Validation loss")
plt.title("Validation loss for a model with insufficient capacity")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
model = nn.Sequential(
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
optimizer = torch.optim.RMSprop(model.parameters())
loss_fn = nn.CrossEntropyLoss()
history_large_model = fit_classifier(
    model, train_images, train_labels, epochs=20, batch_size=128,
    optimizer=optimizer, loss_fn=loss_fn, validation_split=0.2,
)

In [0]:
val_loss = history_large_model.history["val_loss"]
epochs = range(1, 21)
plt.plot(epochs, val_loss, "b-", label="Validation loss")
plt.title("Validation loss for a model with appropriate capacity")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
model = nn.Sequential(
    nn.Linear(28 * 28, 2048),
    nn.ReLU(),
    nn.Linear(2048, 2048),
    nn.ReLU(),
    nn.Linear(2048, 2048),
    nn.ReLU(),
    nn.Linear(2048, 10),
)
optimizer = torch.optim.RMSprop(model.parameters())
loss_fn = nn.CrossEntropyLoss()
history_very_large_model = fit_classifier(
    model, train_images, train_labels, epochs=20, batch_size=32,
    optimizer=optimizer, loss_fn=loss_fn, validation_split=0.2,
)

In [0]:
val_loss = history_very_large_model.history["val_loss"]
epochs = range(1, 21)
plt.plot(epochs, val_loss, "b-", label="Validation loss")
plt.title("Validation loss for a model with too much capacity")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

### Improving generalization

#### Dataset curation

#### Feature engineering

#### Using early stopping

#### Regularizing your model

##### Reducing the network's size

In [ ]:
# PyTorch: there is no keras.datasets.imdb. We reconstruct the same task from the
# HuggingFace `datasets` "imdb" corpus: build a 10000-word vocabulary by
# frequency and integer-encode each review, mirroring
# keras.datasets.imdb.load_data(num_words=10000). The exact word indices differ
# from Keras (it ships a precomputed index), but the multi-hot vectorization and
# the model below are identical. NOTE: substituted dataset source.
from datasets import load_dataset
import re
from collections import Counter

raw = load_dataset("imdb")

def tokenize(text):
    return re.findall(r"[a-z0-9']+", text.lower())

counter = Counter()
for ex in raw["train"]:
    counter.update(tokenize(ex["text"]))
# Reserve indices 0-2 like Keras (pad/start/oov); the top words get the rest.
vocab = {w: i + 3 for i, (w, _) in enumerate(counter.most_common(10000 - 3))}

def encode(split):
    seqs, labels = [], []
    for ex in split:
        ids = [vocab[t] for t in tokenize(ex["text"]) if t in vocab]
        seqs.append(ids)
        labels.append(ex["label"])
    return seqs, np.array(labels, dtype="float32")

train_data, train_labels = encode(raw["train"])

def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension), dtype="float32")
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

train_data = vectorize_sequences(train_data)

# PyTorch: drop the final sigmoid; nn.BCEWithLogitsLoss expects raw logits.
model = nn.Sequential(
    nn.Linear(10000, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
optimizer = torch.optim.RMSprop(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()
history_original = fit_classifier(
    model, train_data, train_labels, epochs=20, batch_size=512,
    optimizer=optimizer, loss_fn=loss_fn, validation_split=0.4, binary=True,
)

In [ ]:
model = nn.Sequential(
    nn.Linear(10000, 4),
    nn.ReLU(),
    nn.Linear(4, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
)
optimizer = torch.optim.RMSprop(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()
history_smaller_model = fit_classifier(
    model, train_data, train_labels, epochs=20, batch_size=512,
    optimizer=optimizer, loss_fn=loss_fn, validation_split=0.4, binary=True,
)

In [0]:
original_val_loss = history_original.history["val_loss"]
smaller_model_val_loss = history_smaller_model.history["val_loss"]
epochs = range(1, 21)
plt.plot(
    epochs,
    original_val_loss,
    "r--",
    label="Validation loss of original model",
)
plt.plot(
    epochs,
    smaller_model_val_loss,
    "b-",
    label="Validation loss of smaller model",
)
plt.title("Original model vs. smaller model (IMDB review classification)")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()
plt.show()

In [ ]:
model = nn.Sequential(
    nn.Linear(10000, 512),
    nn.ReLU(),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, 1),
)
optimizer = torch.optim.RMSprop(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()
history_larger_model = fit_classifier(
    model, train_data, train_labels, epochs=20, batch_size=512,
    optimizer=optimizer, loss_fn=loss_fn, validation_split=0.4, binary=True,
)

In [0]:
original_val_loss = history_original.history["val_loss"]
larger_model_val_loss = history_larger_model.history["val_loss"]
epochs = range(1, 21)
plt.plot(
    epochs,
    original_val_loss,
    "r--",
    label="Validation loss of original model",
)
plt.plot(
    epochs,
    larger_model_val_loss,
    "b-",
    label="Validation loss of larger model",
)
plt.title("Original model vs. larger model (IMDB review classification)")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()
plt.show()

##### Adding weight regularization

In [ ]:
# PyTorch: Keras\' kernel_regularizer=l2(0.002) becomes weight_decay in the
# optimizer. Keras adds l2 * sum(w^2) to the loss; torch\'s RMSprop weight_decay
# adds (weight_decay) * w to the gradient, i.e. it decays toward an equivalent
# 0.5 * weight_decay * w^2 penalty. We pass 0.002 to keep the same coefficient
# the chapter chose (regularization strength is a tunable knob here).
model = nn.Sequential(
    nn.Linear(10000, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
optimizer = torch.optim.RMSprop(model.parameters(), weight_decay=0.002)
loss_fn = nn.BCEWithLogitsLoss()
history_l2_reg = fit_classifier(
    model, train_data, train_labels, epochs=20, batch_size=512,
    optimizer=optimizer, loss_fn=loss_fn, validation_split=0.4, binary=True,
)

In [0]:
original_val_loss = history_original.history["val_loss"]
l2_val_loss = history_l2_reg.history["val_loss"]
epochs = range(1, 21)
plt.plot(
    epochs,
    original_val_loss,
    "r--",
    label="Validation loss of original model",
)
plt.plot(
    epochs,
    l2_val_loss,
    "b-",
    label="Validation loss of L2-regularized model",
)
plt.title(
    "Original model vs. L2-regularized model (IMDB review classification)"
)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()
plt.show()

In [ ]:
# PyTorch: there is no regularizers module. L1 / L1+L2 penalties are added
# manually to the loss, e.g. for an L1 coefficient of 0.001:
#     l1 = 0.001 * sum(p.abs().sum() for p in model.parameters())
#     loss = loss_fn(out, targets) + l1
# and L2 is handled via the optimizer\'s weight_decay (see the cell above).
l1_coeff = 0.001
l1_l2_coeffs = {"l1": 0.001, "l2": 0.001}

##### Adding dropout

In [ ]:
model = nn.Sequential(
    nn.Linear(10000, 16),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(16, 1),
)
optimizer = torch.optim.RMSprop(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()
history_dropout = fit_classifier(
    model, train_data, train_labels, epochs=20, batch_size=512,
    optimizer=optimizer, loss_fn=loss_fn, validation_split=0.4, binary=True,
)

In [0]:
original_val_loss = history_original.history["val_loss"]
dropout_val_loss = history_dropout.history["val_loss"]
epochs = range(1, 21)
plt.plot(
    epochs,
    original_val_loss,
    "r--",
    label="Validation loss of original model",
)
plt.plot(
    epochs,
    dropout_val_loss,
    "b-",
    label="Validation loss of dropout-regularized model",
)
plt.title(
    "Original model vs. dropout-regularized model (IMDB review classification)"
)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()
plt.show()